In [1]:

import pandas as pd
import sys
sys.path.append("..")
from config import REGION_PROCESSED_DIR

# 1. read the CLEAN files
intersections = pd.read_csv(REGION_PROCESSED_DIR / "intersections.csv")
crashes = pd.read_csv(REGION_PROCESSED_DIR / "crashes_clean.csv")

In [2]:
history = crashes[crashes["year"] <= 2022].copy()

# target: 2023 to 2025
target = crashes[(crashes["year"] >= 2023) & (crashes["year"] <= 2025)].copy()

print(len(history), len(target))
print("2026 (excluded):", (crashes["year"] == 2026).sum())


40121 6427
2026 (excluded): 1459


In [3]:
# history: up to end of 2022
history = crashes[crashes["year"] <= 2022].copy()

# target: 2023 to 2025
target = crashes[(crashes["year"] >= 2023) & (crashes["year"] <= 2025)].copy()

print(len(history), len(target))
print("2026 (excluded):", (crashes["year"] == 2026).sum())



40121 6427
2026 (excluded): 1459


In [4]:
# crash counts per intersection (history)
history_counts = history.groupby("intersection_id").agg(
    past_crashes=("crash_id", "count"), #  number of crashes
    past_injured=("injured", "sum"), #  total injured
    past_killed=("killed", "sum"), #  total killed
).reset_index()

# crash counts per intersection (target)
target_counts = target.groupby("intersection_id").agg(
    target_crashes=("crash_id", "count"), 
).reset_index()

#  checks
print(history_counts.head())
print(history_counts["past_crashes"].sum()) 
print(target_counts["target_crashes"].sum()) 


   intersection_id  past_crashes  past_injured  past_killed
0         20006000             1             2          0.0
1         20034000             1             1          0.0
2         20044000             4             4          0.0
3         20046000             1             1          0.0
4         20056000             2             2          0.0
40121
6427


In [ ]:

#    start from ALL intersections, add history counts
train_df = intersections.merge(
        history_counts, how="left", left_on="node", right_on="intersection_id"
)
train_df = train_df.drop(columns=["intersection_id"])   # drop duplicate id

# 2. add target counts
train_df = train_df.merge(
        target_counts, how="left", left_on="node", right_on="intersection_id"
)
train_df = train_df.drop(columns=["intersection_id"])

#    fill zeros for crash columns ONLY (not pci)
count_cols = ["past_crashes", "past_injured", "past_killed", "target_crashes"]
train_df[count_cols] = train_df[count_cols].fillna(0).astype(int)

# 4. checks
print(len(train_df))                                 
print(train_df["node"].is_unique)                     
print(train_df["past_crashes"].sum())                 
print(train_df["target_crashes"].sum())         
print((train_df["target_crashes"] == 0).mean())       
print(train_df["target_crashes"].describe())

# 5. save
train_df.to_csv(REGION_PROCESSED_DIR / "training_table.csv", index=False)

8222
True
40003
6394
0.6931403551447336
count    8222.000000
mean        0.777670
std         1.851091
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        25.000000
Name: target_crashes, dtype: float64
